# Fixing the hierarchy's planner

In `dev_systems.ipynb` (thinking off, 16k tokens) the hierarchy got 25% of the development problems right, against 48% for the single agent and 40% for the ensemble. The planner was the cause: its plans named both requested variables in only 11 of 40 problems, gave one requested variable to each worker in only 6, and set `pass_first` on every problem, so the workers either redid the whole problem or built on a wrong hand-off.

This notebook reruns only the hierarchy, on the same problems at 16k with thinking off, in two variants:

- `thinking_planner`: the planner thinks; workers and merge do not. The proposal allows the orchestrator to be more capable than the workers, and thinking is a way to do that within one model.
- `trace_prompt`: no stage thinks, but the planner gets more specific instructions: trace each requested variable back to the given values, give worker 1 the first and worker 2 the second, list the equations in order, and set `pass_first` only when the second depends on the first.

Run it as natalia@siwek.poznan.pl, whose Drive holds the `dev_systems` results, with **Runtime > Change runtime type > L4 GPU**. Each variant takes about 20 minutes. Results go to `MyDrive/small-multi-agents/results/dev_planner/<variant>/`, each with a `run.log`. If the session disconnects, rerun every cell. Runs resume from the last saved batch.

In [1]:
RUN_NAME = "dev_planner"
BASELINE = "dev_systems"    # its problems and its single agent, ensemble, and hierarchy runs
BUDGETS = [16000]
SAMPLING_SEED = 42
VARIANTS = {
    "thinking_planner": ["--thinking-stages", "planner"],
    "trace_prompt": ["--plan-prompt", "trace"],
}

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [3]:
from google.colab import drive

drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/small-multi-agents/results"
OUT = f"{RESULTS}/{RUN_NAME}"
!mkdir -p {OUT}

Mounted at /content/drive


In [4]:
REPO = "/content/small-multi-agents"
!test -d {REPO} || git clone https://github.com/liatzheng/small-multi-agents.git {REPO}
%cd {REPO}
!git pull --ff-only && git log -1 --format='%H %s'

Cloning into '/content/small-multi-agents'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (180/180), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 180 (delta 109), reused 103 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (180/180), 146.44 KiB | 1.17 MiB/s, done.
Resolving deltas: 100% (109/109), done.
/content/small-multi-agents
Already up to date.
af83517a00ac2d92859d3dec260cab0bfee88e85 Create the log folder in run_logged before writing


Hugging Face downloads use your token if it is stored as a Colab secret: open the key icon in the left sidebar, add a secret named `HF_TOKEN`, and turn on notebook access. The token stays in your Colab account, never in the notebook or the repo.

In [5]:
from notebook_tools import use_hf_token

use_hf_token()

Using the HF_TOKEN Colab secret.


vLLM 0.33.0 brings its own matching torch, torchvision, and transformers, replacing Colab's. The runner runs as a subprocess, so no runtime restart is needed.

In [6]:
!pip install -q -r requirements-vllm.txt
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__, torch.cuda.get_device_name())" | tee {OUT}/environment.txt
!git rev-parse HEAD | tee -a {OUT}/environment.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
vllm 0.33.0 torch 2.14.0+cu130 transformers 5.20.2 NVIDIA L4
af83517a00ac2d92859d3dec260cab0b

## Problems

A copy of the `dev_systems` problems, so this folder holds everything its runs used.

In [7]:
import shutil

PROBLEMS = f"{OUT}/problems.jsonl"
shutil.copyfile(f"{RESULTS}/{BASELINE}/problems.jsonl", PROBLEMS)
!wc -l {PROBLEMS}

40 /content/drive/MyDrive/small-multi-agents/results/dev_planner/problems.jsonl


## Run

The hierarchy with thinking off, one variant at a time. Each variant's output is also appended to its own `run.log`.

In [8]:
from notebook_tools import run_logged

for name, flags in VARIANTS.items():
    run_logged(["python", "run_vllm.py", PROBLEMS, "--out-dir", f"{OUT}/{name}", "--systems", "hierarchy",
                "--budgets", *map(str, BUDGETS), "--seed", str(SAMPLING_SEED), "--no-thinking", *flags,
                "--resume"], f"{OUT}/{name}/run.log")

=== python run_vllm.py /content/drive/MyDrive/small-multi-agents/results/dev_planner/problems.jsonl --out-dir /content/drive/MyDrive/small-multi-agents/results/dev_planner/thinking_planner --systems hierarchy --budgets 16000 --seed 42 --no-thinking --thinking-stages planner --resume
=== started 2026-11-22T21:18:47+00:00
W1122 21:19:00.579000 2106 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1122 21:19:00.579000 2106 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1122 21:19:00.580000 2106 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=2106) INFO 11-22 21:19:06 [startup.py:96] non-default args: {'dtype': 'bfloat16', 'max_model_len': 16000, 'disable_log_stats': True, 'revision': '1cfa9a7208912126459214e8b04321603b3df60c', 'tokenizer_revision': '1cfa9a7208912126459214e8b04321603b3df60c', 'model': 'Qwen/Qwen3-4B'}
(Main

## Results

Each variant next to the `dev_systems` runs at the same budget. `hierarchy` is the default planner from `dev_systems`. `length_limit` counts problems where some stage ran out of tokens, and `protocol_failure` problems that stopped early because the plan was unreadable or a report was empty.

In [9]:
import json
from pathlib import Path

import pandas as pd
from notebook_tools import load_runs


def summarize(df, by):
    return df.groupby(by, sort=False).agg(
        accuracy=("correct", "mean"),
        problems=("correct", "size"),
        wrong_answer=("status", lambda s: (s == "wrong_answer").mean()),
        length_limit=("ran_out", "mean"),
        no_majority=("status", lambda s: (s == "no_majority").mean()),
        protocol_failure=("status", lambda s: s.isin(["invalid_plan", "empty_response"]).mean()),
        mean_total=("total", "mean"),
    ).round(2)


baseline = pd.DataFrame(load_runs(f"{RESULTS}/{BASELINE}"))
frames = [baseline[baseline.budget.isin(BUDGETS)]]
for name in VARIANTS:
    rows = load_runs(f"{OUT}/{name}")
    if rows:
        frames.append(pd.DataFrame(rows).assign(system=f"hierarchy_{name}"))
df = pd.concat(frames)
df.to_csv(f"{OUT}/summary_problems.csv", index=False)
by_system = summarize(df, ["system", "budget"])
by_system.to_csv(f"{OUT}/summary_by_system.csv")
by_system

,,accuracy,problems,wrong_answer,length_limit,no_majority,protocol_failure,mean_total
system,budget,,,,,,,
ensemble,16000,0.40,40,0.20,0.02,0.4,0.0,8822.88
hierarchy,16000,0.25,40,0.65,0.05,0.0,0.0,4531.40
single_agent,16000,0.48,40,0.45,0.02,0.0,0.0,3164.75
hierarchy_thinking_planner,16000,0.28,40,0.60,0.02,0.0,0.0,6136.55
hierarchy_trace_prompt,16000,0.08,40,0.90,0.00,0.0,0.0,4067.58


Accuracy by depth and problem kind.

In [10]:
by_depth = summarize(df, ["system", "budget", "d", "kind"])
by_depth.to_csv(f"{OUT}/summary_by_depth.csv")
by_depth["accuracy"].unstack(["d", "kind"])

d                                           2                     6          
kind                              independent dependent independent dependent
system                     budget                                            
ensemble                   16000          0.7       0.6         0.2       0.1
hierarchy                  16000          0.4       0.4         0.1       0.1
single_agent               16000          0.8       0.6         0.1       0.4
hierarchy_thinking_planner 16000          0.2       0.5         0.1       0.3
hierarchy_trace_prompt     16000          0.0       0.0         0.1       0.2

## Plans

How well each planner split the work. `names_both` is the share of plans that mention both requested variables, `one_each` the share giving one requested variable to each worker, and `pass_first_right` the share whose `pass_first` matches the problem kind (true for dependent, false for independent). `workers_right` is the share where worker 2's report contains both correct values.

In [11]:
import re


def hierarchy_logs():
    yield "hierarchy", Path(f"{RESULTS}/{BASELINE}")
    for name in VARIANTS:
        yield f"hierarchy_{name}", Path(f"{OUT}/{name}")


def read_plan(text):
    found = re.search(r"\{.*\}", text, re.DOTALL)
    try:
        plan = json.loads(found.group(0)) if found else None
    except ValueError:
        return None
    return plan if isinstance(plan, dict) else None


plans, calls = [], []
for system, folder in hierarchy_logs():
    for budget in BUDGETS:
        path = folder / f"b{budget}" / "hierarchy.jsonl"
        if not path.exists():
            continue
        for r in map(json.loads, path.read_text().splitlines()):
            a, b = re.search(r"Return \((v\d+), (v\d+)\)", r["prompt"]).groups()
            plan = read_plan(r["calls"][0]["final_text"]) if r["calls"] else None
            first, second = (str(plan.get("first", "")), str(plan.get("second", ""))) if plan else ("", "")
            worker_2 = r["calls"][2]["final_text"] if len(r["calls"]) > 2 else ""
            plans.append({
                "system": system, "d": r["chain_length"], "kind": r["kind"],
                "readable": plan is not None,
                "names_both": a in first + second and b in first + second,
                "one_each": (a in first and b in second and b not in first)
                            or (b in first and a in second and a not in first),
                "pass_first_right": plan is not None and plan.get("pass_first") == (r["kind"] == "dependent"),
                "workers_right": all(str(v) in worker_2 for v in r["answer"]),
            })
            for call in r["calls"]:
                calls.append({"system": system, "stage": call["stage"], "cap": call["budget"],
                              "output": call["output_tokens"], "stop_reason": call["stop_reason"]})
plans = pd.DataFrame(plans).groupby(["system", "kind"], sort=False).mean(numeric_only=True).drop(columns="d").round(2)
plans.to_csv(f"{OUT}/summary_plans.csv")
plans

readable  names_both  one_each  \
system                     kind                                          
hierarchy                  independent      1.00        0.20      0.05   
                           dependent        1.00        0.35      0.25   
hierarchy_thinking_planner independent      0.95        0.90      0.50   
                           dependent        1.00        1.00      0.85   
hierarchy_trace_prompt     independent      1.00        1.00      1.00   
                           dependent        1.00        1.00      1.00   

                                        pass_first_right  workers_right  
system                     kind                                          
hierarchy                  independent              0.00           0.15  
                           dependent                1.00           0.15  
hierarchy_thinking_planner independent              0.45           0.15  
                           dependent                0.95           0.30  
hierarchy_trace_prompt     independent              0.00           0.05  
                           dependent                1.00           0.10

How each stage spends its tokens. A thinking planner uses far more than the default one, and `ran_out` shows whether its quarter of the budget was enough.

In [12]:
stages = pd.DataFrame(calls).groupby(["system", "stage"], sort=False).agg(
    calls=("output", "size"),
    mean_cap=("cap", "mean"),
    mean_output=("output", "mean"),
    max_output=("output", "max"),
    ran_out=("stop_reason", lambda s: (s != "stop").mean()),
).round(2)
stages.to_csv(f"{OUT}/summary_hierarchy_stages.csv")
stages

calls  mean_cap  mean_output  max_output  \
system                     stage                                                
hierarchy                  planner      40   4000.00        64.80         120   
                           worker_1     40   7466.60       923.55        7015   
                           worker_2     38  10409.58        75.13        1275   
                           merge        38  13273.29       456.89        2670   
hierarchy_thinking_planner planner      40   4000.00      1976.25        3532   
                           worker_1     39   5599.13       599.97        2114   
                           worker_2     39   8550.87       189.54        1576   
                           merge        39  11414.82       343.36        2700   
hierarchy_trace_prompt     planner      40   4000.00       212.68         318   
                           worker_1     40   7258.72       442.75        1466   
                           worker_2     40  10291.10        17.23          20   
                           merge        40  13297.12       270.25        1508   

                                     ran_out  
system                     stage              
hierarchy                  planner      0.00  
                           worker_1     0.05  
                           worker_2     0.00  
                           merge        0.00  
hierarchy_thinking_planner planner      0.02  
                           worker_1     0.00  
                           worker_2     0.00  
                           merge        0.00  
hierarchy_trace_prompt     planner      0.00  
                           worker_1     0.00  
                           worker_2     0.00  
                           merge        0.00

## Reading the results

- A variant that brings `one_each` and `pass_first_right` close to 1 shows the planner can split this task. If its accuracy still trails the single agent, the workers or the hand-off are the next problem; `workers_right` shows which.
- A variant that beats the single agent's 48%, most at depth 6, is the pattern the proposal predicts and the hierarchy to use in the main grid.
- If `thinking_planner` often runs out at the planner stage, its quarter of the budget is too small for thinking at 16k.
- With 10 pairs per depth and kind, treat gaps under about 20 points as noise.